# 45 — NIR passport (LIF graph round-trip)

Build a small **NIR** LIF graph, import it through SC-NeuroCore's bridge,
step it, write/reload, and compare structure. Pedagogy for interop — not a
claim that every framework round-trips bit-exact.

## Honesty box

| | |
|---|---|
| **Proves** | `nir.NIRGraph` → `from_nir` → step → `nir.write` → reload works for this toy LIF graph when `nir` is installed. |
| **Does not prove** | Full catalogue polyglot neurons export as NIR, or Norse/snnTorch identity. |
| **Artefacts** | Tempfile only. |
| **Note** | NIR LIF nodes are framework LIF units, not the polyglot HH/AdEx table. HF catalogue models remain primary elsewhere (NB-41). |


In [ ]:
from __future__ import annotations

import os
import tempfile

import numpy as np

try:
    import nir
except ImportError as exc:
    raise SystemExit(f"nir package required for NB-45: {exc}") from exc

from sc_neurocore.nir_bridge import from_nir

print("NIR", nir.__version__)


In [ ]:
n_in, n_out = 4, 5
rng = np.random.RandomState(0)
nodes = {
    "input": nir.Input(input_type={"input": np.array([n_in])}),
    "affine": nir.Affine(
        weight=rng.randn(n_out, n_in).astype(np.float32) * 0.5,
        bias=np.zeros(n_out, dtype=np.float32),
    ),
    "lif": nir.LIF(
        tau=np.full(n_out, 20.0),
        r=np.ones(n_out),
        v_leak=np.zeros(n_out),
        v_threshold=np.ones(n_out),
    ),
    "output": nir.Output(output_type={"output": np.array([n_out])}),
}
edges = [("input", "affine"), ("affine", "lif"), ("lif", "output")]
graph = nir.NIRGraph(nodes=nodes, edges=edges)
network = from_nir(graph)
print(network.summary())


In [ ]:
n_steps = 80
spikes = []
x = np.ones(n_in, dtype=np.float32)
network.reset()
for t in range(n_steps):
    drive = x * (1.0 + 0.2 * np.sin(2 * np.pi * t / 20.0))
    out = network.step({"input": drive})
    spikes.append(np.asarray(out["output"], dtype=float).copy())
spikes = np.stack(spikes, axis=0)
print("spike array", spikes.shape, "events", float(spikes.sum()))

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 3))
    for j in range(n_out):
        times = np.where(spikes[:, j] > 0)[0]
        ax.scatter(times, np.full_like(times, j), marker="|", s=80, linewidths=0.8)
    ax.set_title("NIR LIF passport — spike raster")
    ax.set_xlabel("step")
    ax.set_ylabel("neuron")
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib missing — skip plot")


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "passport.nir")
    nir.write(path, graph)
    size = os.path.getsize(path)
    reloaded = from_nir(path)
    network.reset()
    reloaded.reset()
    a = network.step({"input": np.ones(n_in)})
    b = reloaded.step({"input": np.ones(n_in)})
    print(f"wrote {size} bytes; reload nodes={len(reloaded.nodes)} edges={len(reloaded.edges)}")
    print("out original", np.asarray(a["output"]))
    print("out reloaded", np.asarray(b["output"]))
print("NB-45 complete.")
